# Supervised Model Evaluation and Calibration (M2/M3/M5)

This notebook uses reusable repository code. Search and calibration decisions use training/validation folds only; the final test fixture is evaluated once for smoke evidence. Full ISOT/WELFake benchmark claims require raw datasets.

In [1]:
import sys
from pathlib import Path
root_dir = Path('..').resolve() if Path('../src').exists() else Path('.').resolve()
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

import numpy as np
import pandas as pd
from sklearn.model_selection import learning_curve, validation_curve

from src.evaluation.metrics import (
    evaluate_predictions, mcnemar_test, paired_bootstrap_regression, regression_metrics
)
from src.evaluation.plots import (
    plot_learning_curve, plot_reliability_comparison, plot_roc_pr, plot_validation_curve
)
from src.features.text import TfidfTextPipeline
from src.models.classical import build_logistic_model, build_random_forest

train_path = root_dir / 'tests' / 'fixtures' / 'train.csv'
test_path = root_dir / 'tests' / 'fixtures' / 'test.csv'
train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
tfidf = TfidfTextPipeline(min_df=1, max_df=1.0, max_features=200)
X_train = tfidf.fit_transform(train['content'])
X_test = tfidf.transform(test['content'])
y_train, y_test = train['label'].to_numpy(), test['label'].to_numpy()
baseline = build_logistic_model('l2', max_iter=500).fit(X_train, y_train)
forest = build_random_forest(n_estimators=20, random_state=42).fit(X_train.toarray(), y_train)
baseline_proba = baseline.predict_proba(X_test)
forest_proba = forest.predict_proba(X_test.toarray())
print(evaluate_predictions(y_test, baseline_proba).to_dict())


{'accuracy': 1.0, 'precision': 1.0, 'recall': 1.0, 'f1_macro': 1.0, 'f1_weighted': 1.0, 'roc_auc': 1.0, 'pr_auc': 1.0, 'brier_score': 0.001355738327420467, 'confusion_matrix': [[2, 0], [0, 2]]}


In [2]:
root_dir = Path('..').resolve() if Path('../src').exists() else Path('.').resolve()
report_dir = root_dir / 'reports' / 'evaluation'
report_dir.mkdir(parents=True, exist_ok=True)
plot_roc_pr(y_test, baseline_proba, report_dir / 'notebook_roc_pr.png', label='logistic')
plot_reliability_comparison(y_test, {'logistic': baseline_proba, 'forest': forest_proba}, report_dir / 'notebook_reliability.png')


WindowsPath('reports/evaluation/notebook_reliability.png')

In [3]:
sizes, train_scores, validation_scores = learning_curve(
    build_logistic_model('l2', max_iter=500), X_train, y_train, cv=2, scoring='accuracy',
    train_sizes=np.asarray([0.5, 0.75, 1.0]),
)
plot_learning_curve(sizes, train_scores, validation_scores, report_dir / 'notebook_learning_curve.png')


WindowsPath('reports/evaluation/notebook_learning_curve.png')

In [4]:
param_values = [0.25, 1.0, 4.0]
train_scores, validation_scores = validation_curve(
    build_logistic_model('l2', max_iter=500), X_train, y_train, param_name='classifier__C',
    param_range=param_values, cv=2, scoring='accuracy',
)
plot_validation_curve(param_values, train_scores, validation_scores, report_dir / 'notebook_validation_curve.png', parameter_name='C')


WindowsPath('reports/evaluation/notebook_validation_curve.png')

In [5]:
# Nested CV is implemented in src.evaluation.metrics; pass a fold-local search factory in a production run.
print('Nested-CV contract: nested_stratified_cross_validate')


Nested-CV contract: nested_stratified_cross_validate


In [6]:
print(mcnemar_test(y_test, baseline_proba, forest_proba))
actual = np.asarray([1.0, 2.0, 3.0, 4.0])
pred_a, pred_b = np.asarray([1.1, 1.9, 3.2, 3.8]), np.asarray([1.5, 2.5, 2.5, 3.5])
print(regression_metrics(actual, pred_a))
print(paired_bootstrap_regression(actual, pred_a, pred_b, n_bootstrap=100, random_state=42))


{'b_model_a_only_correct': 0, 'c_model_b_only_correct': 0, 'discordant_pairs': 0, 'statistic_continuity_corrected': 0.0, 'continuity_corrected_p_value': 1.0, 'exact_binomial_p_value': 1.0, 'p_value': 1.0, 'interpretation': 'No evidence of a difference at alpha=0.05'}
{'rmse': 0.1581138830084191, 'mae': 0.15000000000000013, 'mape': 6.666666666666672, 'r2': 0.98}
{'metric': 'rmse', 'observed_difference_a_minus_b': -0.3418861169915809, 'ci_low': -0.3999999999999999, 'ci_high': -0.2999999999999998, 'confidence_level': 0.95, 'n_bootstrap': 100, 'random_state': 42}


In [7]:
try:
    from src.models.classical import shap_values
    feature_names = np.asarray(tfidf.get_feature_names(), dtype=object)
    print(shap_values(forest, X_train.toarray(), feature_names, max_samples=100))
except Exception as exc:
    print(f'SHAP analysis note: {exc}')


                feature  mean_abs_shap
0                  real       0.145831
1                  fake       0.049809
2                  hoax       0.023890
3    reviewed statement       0.021793
4                 story       0.018324
..                  ...            ...
159              update       0.000000
160       update public       0.000000
161            verified       0.000000
162     verified report       0.000000
163             weather       0.000000

[164 rows x 2 columns]
